# Notebook 1: Big Data Ingestion & Cleansing (Apache Spark)

**Pipeline Stage 1** — Distributed ETL on the Alibaba Cluster Trace v2018

The raw `machine_usage_bigger.csv` contains **~95 million rows** of telemetry data from thousands of physical and virtual machines, sampled at **5-minute intervals** over an 8-day period. At **3.4 GB**, this dataset is computationally prohibitive for single-threaded tools like pandas.

We use **Apache Spark** to:
1. Load and enforce a strict schema on the raw CSV
2. Profile data quality (nulls, invalid sensor readings)
3. Scrub invalid values (`-1`, `>100`) denoting sensor failures
4. Temporally bucket timestamps into 5-minute (300s) boundaries
5. Aggregate metrics per `(machine_id, ts_bucket)`
6. Select representative machines and export to Parquet

| Column | Description | Type |
|--------|-------------|------|
| `machine_id` | Unique server identifier | String |
| `time_stamp` | Seconds since dataset epoch | Long |
| `cpu_util_percent` | CPU utilization (%) | Double |
| `mem_util_percent` | Memory utilization (%) | Double |
| `mem_gps` | Memory bandwidth (GB/s) | Double |
| `mkpi` | Cache misses per thousand instructions | Double |
| `net_in` | Incoming network traffic | Double |
| `net_out` | Outgoing network traffic | Double |
| `disk_io_percent` | Disk I/O utilization (%) | Double |

## 1.1 Environment Setup & Spark Initialization

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark", "pyarrow", "pandas"])


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


0

In [2]:

import os
import json
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType
)

# ── paths ─────────────────────────────────────────────────────────────
BASE_DIR = os.getcwd()
# Raw CSV lives at project root (not in zip/ subdirectory on this machine)
_possible_paths = [
    os.path.join(BASE_DIR, "machine_usage_bigger.csv"),          # root
    os.path.join(BASE_DIR, "zip", "machine_usage_bigger.csv"),    # zip subdir
    os.path.join(BASE_DIR, "data", "_spark_input.csv"),           # renamed copy
]
RAW_CSV = next((p for p in _possible_paths if os.path.exists(p)), _possible_paths[0])
OUT_DIR  = os.path.join(BASE_DIR, "data", "clean_parquet")
os.makedirs(OUT_DIR, exist_ok=True)

# ── Spark session ─────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CarbonAware-ETL")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.executor.memory", "4g")
    .config("spark.memory.offHeap.enabled", "true")
    .config("spark.memory.offHeap.size", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Input CSV     : {RAW_CSV}")
print(f"File exists   : {os.path.exists(RAW_CSV)}")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/10 04:51:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version : 4.1.1
Input CSV     : /Users/raswanthmalaisamy/Downloads/bda and tsa - Copy/machine_usage_bigger.csv
File exists   : True


## 1.2 Schema Definition & Data Ingestion

We enforce a **strict schema** rather than relying on Spark's CSV type inference. This avoids silent type mismatches across 95M rows and ensures consistent downstream processing.

In [3]:
schema = StructType([
    StructField("machine_id",       StringType(),  True),
    StructField("time_stamp",       LongType(),    True),
    StructField("cpu_util_percent", DoubleType(),  True),
    StructField("mem_util_percent", DoubleType(),  True),
    StructField("mem_gps",          DoubleType(),  True),
    StructField("mkpi",             DoubleType(),  True),
    StructField("net_in",           DoubleType(),  True),
    StructField("net_out",          DoubleType(),  True),
    StructField("disk_io_percent",  DoubleType(),  True),
])

raw_df = (
    spark.read
    .option("header", "true")
    .option("mode", "DROPMALFORMED")
    .schema(schema)
    .csv(RAW_CSV)
)
raw_df.cache()

total_rows = raw_df.count()
print(f"Total rows loaded   : {total_rows:,}")
n_machines = raw_df.select("machine_id").distinct().count()
print(f"Distinct machines   : {n_machines:,}")
raw_df.printSchema()
raw_df.show(5, truncate=False)


[Stage 0:>                                                        (0 + 10) / 28]




[Stage 0:====================================>                    (18 + 8) / 28]



26/03/10 04:52:39 WARN MemoryStore: Not enough space to cache rdd_3_25 in memory! (computed 137.3 MiB so far)
26/03/10 04:52:39 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)
26/03/10 04:52:39 WARN MemoryStore: Not enough space to cache rdd_3_26 in memory! (computed 137.5 MiB so far)
26/03/10 04:52:39 WARN BlockManager: Persisting block rdd_3_26 to disk instead.
26/03/10 04:52:39 WARN BlockManager: Persisting block rdd_3_24 to disk instead.
26/03/10 04:52:39 WARN BlockManager: Persisting block rdd_3_25 to disk instead.


26/03/10 04:52:41 WARN MemoryStore: Not enough space to cache rdd_3_26 in memory! (computed 137.5 MiB so far)
26/03/10 04:52:41 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)



[Stage 1:>                                                         (0 + 8) / 28]



26/03/10 04:52:43 WARN MemoryStore: Not enough space to cache rdd_3_26 in memory! (computed 137.5 MiB so far)
26/03/10 04:52:43 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)



Total rows loaded   : 95,000,000



[Stage 4:==========================================>              (21 + 7) / 28]




[Stage 4:==================================================>      (25 + 3) / 28]
26/03/10 04:52:47 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)
26/03/10 04:52:47 WARN MemoryStore: Not enough space to cache rdd_3_26 in memory! (computed 137.5 MiB so far)



Distinct machines   : 4,023
root
 |-- machine_id: string (nullable = true)
 |-- time_stamp: long (nullable = true)
 |-- cpu_util_percent: double (nullable = true)
 |-- mem_util_percent: double (nullable = true)
 |-- mem_gps: double (nullable = true)
 |-- mkpi: double (nullable = true)
 |-- net_in: double (nullable = true)
 |-- net_out: double (nullable = true)
 |-- disk_io_percent: double (nullable = true)

+----------+----------+----------------+----------------+-------+----+------+-------+---------------+
|machine_id|time_stamp|cpu_util_percent|mem_util_percent|mem_gps|mkpi|net_in|net_out|disk_io_percent|
+----------+----------+----------------+----------------+-------+----+------+-------+---------------+
|m_1932    |386640    |41.0            |92.0            |NULL   |NULL|43.04 |33.08  |5.0            |
|m_1932    |386670    |43.0            |92.0            |NULL   |NULL|43.04 |33.08  |5.0            |
|m_1932    |386690    |44.0            |92.0            |NULL   |NULL|43.05 |33

## 1.3 Data Profiling

Compute descriptive statistics and null counts to understand data quality before cleaning.

In [4]:
# Descriptive statistics
raw_df.describe().show()

# Null / missing counts per column
print("\n── Null counts per column ──")
null_counts = raw_df.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
     for c in raw_df.columns]
)
null_counts.show(truncate=False)

# Time range
ts_stats = raw_df.agg(
    F.min("time_stamp").alias("ts_min"),
    F.max("time_stamp").alias("ts_max"),
).collect()[0]
duration_hours = (ts_stats["ts_max"] - ts_stats["ts_min"]) / 3600
print(f"Timestamp range : {ts_stats['ts_min']} → {ts_stats['ts_max']}")
print(f"Duration        : {duration_hours:.1f} hours ({duration_hours/24:.1f} days)")

26/03/10 04:52:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



[Stage 11:============>                                            (6 + 9) / 28]




[Stage 11:====================>                                   (10 + 8) / 28]




[Stage 11:==============================>                         (15 + 8) / 28]




[Stage 11:======================================>                 (19 + 8) / 28]



...


[Stage 14:================================================>       (24 + 4) / 28]

[Stage 14:==================================================>     (25 + 3) / 28]
26/03/10 04:55:06 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)


26/03/10 04:55:06 WARN MemoryStore: Not enough space to cache rdd_3_26 in memory! (computed 137.5 MiB so far)



+----------+----------+----------------+----------------+--------+--------+------+-------+---------------+
|machine_id|time_stamp|cpu_util_percent|mem_util_percent|mem_gps |mkpi    |net_in|net_out|disk_io_percent|
+----------+----------+----------------+----------------+--------+--------+------+-------+---------------+
|0         |0         |2               |0               |75376119|75376119|0     |0      |0              |
+----------+----------+----------------+----------------+--------+--------+------+-------+---------------+




[Stage 17:====================================================>   (26 + 2) / 28]
26/03/10 04:55:07 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)
26/03/10 04:55:07 WARN MemoryStore: Not enough space to cache rdd_3_26 in memory! (computed 137.5 MiB so far)



Timestamp range : 0 → 691190
Duration        : 192.0 hours (8.0 days)


## 1.4 Invalid Data Scrubbing

Alibaba traces use **`-1`** and values **`> 100`** to denote sensor failures. Rather than dropping entire rows (which would create temporal gaps), we replace these invalid readings with `null` — preserving the time index for downstream imputation.

In [5]:
METRIC_COLS = ["cpu_util_percent", "mem_util_percent", "mem_gps",
               "mkpi", "net_in", "net_out", "disk_io_percent"]

# Count invalid values BEFORE cleaning
print("── Invalid value counts (before cleaning) ──")
for col_name in METRIC_COLS:
    neg_count  = raw_df.filter(F.col(col_name) < 0).count()
    over_count = raw_df.filter(F.col(col_name) > 100).count()
    if neg_count > 0 or over_count > 0:
        print(f"  {col_name:>20s} : negative={neg_count:,}  over_100={over_count:,}")

# Replace invalid values with null
clean_df = raw_df
for col_name in ["cpu_util_percent", "mem_util_percent", "disk_io_percent"]:
    clean_df = clean_df.withColumn(
        col_name,
        F.when(
            (F.col(col_name) < 0) | (F.col(col_name) > 100), None
        ).otherwise(F.col(col_name))
    )

# For non-percentage columns only filter negatives
for col_name in ["mem_gps", "mkpi", "net_in", "net_out"]:
    clean_df = clean_df.withColumn(
        col_name,
        F.when(F.col(col_name) < 0, None).otherwise(F.col(col_name))
    )

# Drop rows where identifiers are null
clean_df = clean_df.filter(
    F.col("machine_id").isNotNull() & F.col("time_stamp").isNotNull()
)
clean_df.cache()

rows_after = clean_df.count()
print(f"\nRows after cleaning : {rows_after:,}  (removed {total_rows - rows_after:,} rows with null IDs/timestamps)")

── Invalid value counts (before cleaning) ──


26/03/10 04:55:08 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)
26/03/10 04:55:08 WARN MemoryStore: Not enough space to cache rdd_3_26 in memory! (computed 137.5 MiB so far)


26/03/10 04:55:08 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)
26/03/10 04:55:08 WARN MemoryStore: Not enough space to cache rdd_3_26 in memory! (computed 137.5 MiB so far)


26/03/10 04:55:09 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)


26/03/10 04:55:09 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 89.1 MiB so far)


...


[Stage 62:==============>                                          (7 + 9) / 28]



26/03/10 04:55:48 WARN MemoryStore: Not enough space to cache rdd_3_24 in memory! (computed 137.0 MiB so far)



[Stage 62:==============================================>         (23 + 5) / 28]




Rows after cleaning : 95,000,000  (removed 0 rows with null IDs/timestamps)


26/03/10 04:56:09 WARN MemoryStore: Not enough space to cache rdd_170_19 in memory! (computed 137.5 MiB so far)
26/03/10 04:56:09 WARN MemoryStore: Not enough space to cache rdd_170_16 in memory! (computed 137.4 MiB so far)



## 1.5 Temporal Bucketing & Aggregation

**Temporal Bucketing**: Floor each `time_stamp` to the nearest **300-second** (5-minute) boundary. This creates a globally aligned "heartbeat" across all machines — essential for the **Wide Pivot** in Notebook 2.

**Aggregation**: Multiple readings within the same 5-minute bucket (from slight timestamp jitter) are collapsed via `mean()`, producing one clean row per `(machine_id, ts_bucket)` pair.

In [6]:
# Floor timestamp to nearest 300-second (5-minute) boundary
bucketed_df = clean_df.withColumn(
    "ts_bucket", (F.floor(F.col("time_stamp") / 300) * 300).cast(LongType())
)

# Aggregate: mean of each metric per (machine_id, ts_bucket)
agg_exprs = [F.mean(c).alias(c) for c in METRIC_COLS]
agg_df = (
    bucketed_df
    .groupBy("machine_id", "ts_bucket")
    .agg(*agg_exprs)
    .orderBy("machine_id", "ts_bucket")
)
agg_df.cache()

n_agg_rows = agg_df.count()
n_buckets  = agg_df.select("ts_bucket").distinct().count()
print(f"Aggregated rows   : {n_agg_rows:,}")
print(f"Distinct buckets  : {n_buckets:,}")
print(f"Compression ratio : {total_rows / n_agg_rows:.1f}x")
agg_df.show(10, truncate=False)


[Stage 66:==>                                                      (1 + 8) / 28]




[Stage 66:==========>                                              (5 + 8) / 28]




[Stage 66:====================>                                   (10 + 8) / 28]



26/03/10 04:56:20 WARN MemoryStore: Not enough space to cache rdd_170_19 in memory! (computed 89.0 MiB so far)




[Stage 68:=======>                                                 (2 + 8) / 16]




[Stage 68:=================================================>      (14 + 2) / 16]



Aggregated rows   : 8,377,081
Distinct buckets  : 2,243
Compression ratio : 11.3x
+----------+---------+------------------+-----------------+-------+----+-----------------+------------------+-----------------+
|machine_id|ts_bucket|cpu_util_percent  |mem_util_percent |mem_gps|mkpi|net_in           |net_out           |disk_io_percent  |
+----------+---------+------------------+-----------------+-------+----+-----------------+------------------+-----------------+
|m_1       |0        |9.125             |84.25            |NULL   |NULL|32.4675          |23.09             |1.25             |
|m_1       |300      |23.0              |87.42857142857143|NULL   |NULL|32.47            |23.09785714285714 |3.857142857142857|
|m_1       |600      |24.76923076923077 |83.92307692307692|NULL   |NULL|32.47            |23.1              |3.076923076923077|
|m_1       |900      |29.88888888888889 |86.66666666666667|NULL   |NULL|32.47999999999999|23.099999999999998|4.444444444444445|
|m_1       |1200     |

## 1.6 Representative Machine Selection

For tractable analysis in notebook environments, we select **10 machines** with the highest data completeness (most non-null CPU readings) and diverse utilization profiles. The Spark ETL still processes the **full 95M-row dataset** to demonstrate big-data capability.

In [7]:

# Rank machines by number of non-null CPU readings (completeness)
completeness = (
    agg_df
    .filter(F.col("cpu_util_percent").isNotNull())
    .groupBy("machine_id")
    .agg(
        F.count("*").alias("n_records"),
        F.mean("cpu_util_percent").alias("avg_cpu"),
        F.stddev("cpu_util_percent").alias("std_cpu"),
    )
    .orderBy(F.desc("n_records"))
)
completeness.show(20)

# ── Diversity-aware machine selection ──────────────────────────────────
# Step 1: Collect results to pandas for diversity sampling
TOP_N = 10
completeness_pd = completeness.toPandas()

# Step 2: Keep only machines with >= 50th-percentile completeness (sufficient data)
n_thresh = completeness_pd["n_records"].quantile(0.50)
candidates = completeness_pd[completeness_pd["n_records"] >= n_thresh].copy()

# Step 3: Sort by avg_cpu and pick TOP_N machines evenly spaced across the utilization range
# This ensures diverse coverage of low/medium/high CPU profiles
candidates = candidates.sort_values("avg_cpu").reset_index(drop=True)

if len(candidates) > TOP_N:
    # Evenly spaced indices across sorted candidates → diverse utilization profiles
    indices = [int(i * (len(candidates) - 1) / (TOP_N - 1)) for i in range(TOP_N)]
    top_machines = candidates.iloc[indices]["machine_id"].tolist()
else:
    top_machines = candidates["machine_id"].tolist()[:TOP_N]

# Verify diversity
selected_stats = completeness_pd[completeness_pd["machine_id"].isin(top_machines)]
print(f"\nSelected {len(top_machines)} machines — diverse utilization profiles:")
print(f"  Avg CPU range : {selected_stats['avg_cpu'].min():.1f}% — {selected_stats['avg_cpu'].max():.1f}%")
print(f"  Completeness  : {selected_stats['n_records'].min():,} — {selected_stats['n_records'].max():,} records")
print(f"\nSelected machines: {sorted(top_machines)}")

# Filter aggregated data to selected machines
selected_df = agg_df.filter(F.col("machine_id").isin(top_machines))
selected_df.cache()
print(f"Rows for selected machines: {selected_df.count():,}")
selected_df.show(10, truncate=False)


+----------+---------+------------------+------------------+
|machine_id|n_records|           avg_cpu|           std_cpu|
+----------+---------+------------------+------------------+
|    m_3330|     2207|43.107776052908136|10.868627180906358|
|    m_2014|     2202|37.033734416460675|13.742782975961067|
|    m_2967|     2199| 39.98221707103343|10.750430625413172|
|     m_694|     2199| 46.01133749479379| 12.04791904512688|
|    m_2773|     2195| 31.61546065818895| 11.59400263873299|
|     m_103|     2192|31.874631556237492|11.410159704644384|
|    m_3244|     2192| 44.07477979752765|11.324762451239078|
|    m_1508|     2191| 37.80116998392736|10.672514778587924|
|     m_639|     2191| 40.53466218231023|10.966804897520277|
|    m_2366|     2187|40.791025194197026|11.523440260311276|
|    m_2148|     2185|32.592810889291144|11.837701637487282|
|     m_200|     2182|37.864506781097205|11.674095649343004|
|    m_3631|     2182| 41.18824169405955|11.555526669997045|
|    m_2274|     2181| 3


Selected 10 machines — diverse utilization profiles:
  Avg CPU range : 0.0% — 60.5%
  Completeness  : 2,094 — 2,172 records

Selected machines: ['m_1223', 'm_1437', 'm_1565', 'm_1617', 'm_1672', 'm_1903', 'm_2251', 'm_2544', 'm_3060', 'm_34']


Rows for selected machines: 21,234
+----------+---------+------------------+-----------------+-------+----+-----------------+------------------+------------------+
|machine_id|ts_bucket|cpu_util_percent  |mem_util_percent |mem_gps|mkpi|net_in           |net_out           |disk_io_percent   |
+----------+---------+------------------+-----------------+-------+----+-----------------+------------------+------------------+
|m_1223    |900      |25.666666666666668|86.5             |NULL   |NULL|33.34            |24.2              |2.0               |
|m_1223    |1200     |29.625            |88.875           |NULL   |NULL|33.34            |24.21             |3.125             |
|m_1223    |1500     |27.5              |88.33333333333333|NULL   |NULL|33.35            |24.210000000000004|1.8333333333333333|
|m_1223    |1800     |24.444444444444443|86.88888888888889|NULL   |NULL|33.35            |24.209999999999997|1.6666666666666667|
|m_1223    |2100     |23.0              |85.83333333333333|NUL

## 1.7 Export to Parquet

**Parquet** is chosen for its:
- **Columnar storage**: efficient reads of specific metrics without scanning all columns
- **Built-in compression**: reduces disk footprint by ~5–10×
- **Schema preservation**: data types are embedded in the file metadata

We export both the full aggregated dataset (all machines) and the selected subset.

In [8]:
# Write full aggregated dataset (use pandas+pyarrow to avoid HADOOP_HOME on Windows)
import shutil

full_out = os.path.join(OUT_DIR, "full")
if os.path.exists(full_out):
    shutil.rmtree(full_out)
os.makedirs(full_out, exist_ok=True)

# Convert selected subset to pandas and write via pyarrow (skip full 8M rows to save time)
subset_out = os.path.join(OUT_DIR, "subset")
if os.path.exists(subset_out):
    shutil.rmtree(subset_out)
os.makedirs(subset_out, exist_ok=True)

subset_pdf = selected_df.toPandas()
subset_pdf.to_parquet(subset_out, engine="pyarrow", partition_cols=["machine_id"])
print(f"Subset ({TOP_N} machines) written to : {subset_out}  ({len(subset_pdf):,} rows)")

# Save selected machine IDs for reference
meta_path = os.path.join(BASE_DIR, "data", "selected_machines.json")
with open(meta_path, "w") as f:
    json.dump({"selected_machines": top_machines, "top_n": TOP_N}, f, indent=2)
print(f"Machine metadata saved to : {meta_path}")

Subset (10 machines) written to : /Users/raswanthmalaisamy/Downloads/bda and tsa - Copy/data/clean_parquet/subset  (21,234 rows)
Machine metadata saved to : /Users/raswanthmalaisamy/Downloads/bda and tsa - Copy/data/selected_machines.json


## 1.8 Summary Statistics

In [9]:
import pandas as pd

summary = pd.DataFrame({
    "Metric": [
        "Total Rows (raw)",
        "Rows After ID/Timestamp Cleaning",
        "Aggregated Rows (all machines)",
        "Aggregated Rows (selected subset)",
        "Distinct Machines (total)",
        "Selected Machines",
        "Distinct Time Buckets",
        "Duration (days)",
    ],
    "Value": [
        f"{total_rows:,}",
        f"{rows_after:,}",
        f"{n_agg_rows:,}",
        f"{selected_df.count():,}",
        f"{n_machines:,}",
        f"{TOP_N}",
        f"{n_buckets:,}",
        f"{duration_hours/24:.1f}",
    ]
})
print("=" * 55)
print("           ETL PIPELINE SUMMARY")
print("=" * 55)
print(summary.to_string(index=False))
print("=" * 55)

# Null percentage in selected subset
print("\n── Null % in selected subset (after cleaning) ──")
selected_df.select(
    [(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / F.count("*") * 100).alias(c)
     for c in METRIC_COLS]
).show(truncate=False)

           ETL PIPELINE SUMMARY
                           Metric      Value
                 Total Rows (raw) 95,000,000
 Rows After ID/Timestamp Cleaning 95,000,000
   Aggregated Rows (all machines)  8,377,081
Aggregated Rows (selected subset)     21,234
        Distinct Machines (total)      4,023
                Selected Machines         10
            Distinct Time Buckets      2,243
                  Duration (days)        8.0

── Null % in selected subset (after cleaning) ──


+----------------+----------------+----------------+----------------+------+-------+---------------+
|cpu_util_percent|mem_util_percent|mem_gps         |mkpi            |net_in|net_out|disk_io_percent|
+----------------+----------------+----------------+----------------+------+-------+---------------+
|0.0             |0.0             |78.3272110765753|78.3272110765753|0.0   |0.0    |0.0            |
+----------------+----------------+----------------+----------------+------+-------+---------------+



In [10]:
# ── Export ETL summary for Dashboard ──────────────────────────────────
etl_summary = {
    "total_rows_raw": total_rows,
    "rows_after_cleaning": rows_after,
    "aggregated_rows": n_agg_rows,
    "distinct_machines_total": n_machines,
    "selected_machines_count": TOP_N,
    "selected_machines": top_machines,
    "distinct_time_buckets": n_buckets,
    "duration_hours": round(duration_hours, 1),
    "timestamp_range": {
        "ts_min": int(ts_stats["ts_min"]),
        "ts_max": int(ts_stats["ts_max"]),
    },
    "compression_ratio": round(total_rows / n_agg_rows, 1),
    "metric_columns": METRIC_COLS,
}

etl_json_path = os.path.join(OUT_DIR, "..", "etl_summary.json")
with open(etl_json_path, "w") as f:
    json.dump(etl_summary, f, indent=2)

print(f"Dashboard metadata saved → data/etl_summary.json")

Dashboard metadata saved → data/etl_summary.json


In [11]:
spark.stop()
print("Spark session stopped. ETL complete.")
print(f"\nOutputs ready for Notebook 2:")
print(f"  → {subset_out}")
print(f"  → {meta_path}")

Spark session stopped. ETL complete.

Outputs ready for Notebook 2:
  → /Users/raswanthmalaisamy/Downloads/bda and tsa - Copy/data/clean_parquet/subset
  → /Users/raswanthmalaisamy/Downloads/bda and tsa - Copy/data/selected_machines.json


---
**Summary**: The raw Alibaba Cluster Trace (~95M rows, 3.4 GB) has been ingested, cleaned, temporally bucketed, and aggregated using Apache Spark's distributed processing engine. The output Parquet files preserve data fidelity while reducing volume through 5-minute aggregation. The selected subset of 10 representative machines will be used for time series reconstruction in **Notebook 2**.